<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Modeling Workflow (Walkthrough) - Solution

---

### About
Model data from scratch: gather data, clean it, explore it, and then evaluate several models.

### Learning Objectives
- Gather, clean, explore and model a dataset from scratch.
- Split data into testing and training sets using both train/test split and cross-validation and apply both techniques to score a model.
- Evaluate several models.


### Notebook Guide

- Importing libaries
- Load the Data
- Data Cleaning:
     - Initial check
     - Clean up PhD column
     - Drop `NaN`s
- Feature engineering: Binarize `Private` column
- EDA:
     - `apps`
     - What else do we want to explore?
     - Create Histograms of All Numerical Columns
     - Plot a Heatmap of the Correlation Matrix
     - Use seaborn's `.pairplot()` method to create scatterplots
- Model Prep:
     - Create our features matrix (`X`) and target vector (`y`)
     - Train/test split
     - Scaling
     - Instantiate our model
- Evaluation
- LINE Assumptions

# Importing libaries
---

We'll need the following libraries for today's lesson:

1. `pandas`
2. `numpy`
3. `seaborn`
4. `matplotlib.pyplot`
4. `train_test_split` and `cross_val_score` from `sklearn`'s `model_selection` module
5. `LinearRegression`, `LassoCV` and `RidgeCV` from `sklearn`'s `linear_model` module
6. `StandardScaler` from `sklearn`'s `preprocessing` module
7. `mean_squared_error` from `sklearn`'s `metrics` module 

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Load the Data

---

Today's [data set](http://www-bcf.usc.edu/~gareth/ISL/data.html) (`College.csv`) is from the [ISLR website](https://cran.r-project.org/web/packages/ISLR/ISLR.pdf) on page 5. 

Rename `Unnamed: 0` to `University`.

In [ ]:
df = pd.read_csv('../data/College.csv')
df.rename(columns={'Unnamed: 0': 'University'}, inplace=True)
df.head()

 Lowercase and snakecase all column names.

In [ ]:
df.columns = df.columns.str.lower().str.replace('.', '_')
df.head()

In [ ]:
df.info()

# Data cleaning: Initial check
---

Check the following in the cells below:
1. Do we have any null values?
2. Are any numerical columns being read in as `object`?

In [ ]:
# Check for nulls
df.isnull().sum()

In [ ]:
# Check column data types
df.dtypes

In [ ]:
# Hmm... PHD

df['phd'].value_counts()

# Data cleaning: Clean up `PhD` column
---

`PhD` is being read in as a string because some of the cells contain non-numerical values. In the cell below, replace any non-numerical values with `NaN`'s, and change the column datatype to float.

In [ ]:
df['phd'] = df['phd'].replace('?', np.nan).astype(float)

#df['phd'] = df['phd'].map(lambda phd: np.nan if phd == '?' else float(phd))
#df['phd'] = np.where(df['phd'] == '?', np.nan, df['phd']).astype(float)
#df.loc[~df.phd.str.isdigit(), 'phd'] = np.nan


In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

# Data cleaning: Drop `NaN`s
---

Since there are a small percentage of null cells, let's go ahead and drop them.

In [ ]:
df.shape

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.shape

# Feature engineering: Binarize `Private` column
---

In the cells below, convert the `Private` column into numerical values.

In [ ]:
df['private'].value_counts()

In [ ]:
df['private'] = df['private'].map({'Yes': 1, 'No': 0})
# df['private'] = df['private'].map({'Yes': 1, 'No': 0})
# df['private'] = df['private'].apply({'Yes': 1, 'No': 0})
# df['private'] = df['private'].map(lambda s: int(s == 'Yes'))
# df['private'] = np.where(df['private'] == 'Yes', 1, 0)

# EDA: `apps`

In [ ]:
sns.boxplot(x = df['apps'], color = 'cornflowerblue');

In [ ]:
df[df['apps'] > 30000]

In [ ]:
df.shape

In [ ]:
# We can make a decision to drop the outlier, if we want! Test it!

df = df[df['apps'] < 30000]
df.shape

# EDA: What else do we want to explore?
---

In [ ]:
df.describe()

In [ ]:
# phd & grad rate minimum

df['phd'] = np.minimum(df['phd'], 100)
df['grad_rate'] = np.minimum(df['grad_rate'], 100)

In [ ]:
sns.boxplot(x = df['apps'], color = 'cornflowerblue');

In [ ]:
sns.histplot(df['apps'], bins = 25, color = 'cornflowerblue', edgecolor = 'black');

In [ ]:
df.shape

In [ ]:
df[df['apps'] > 30000]

In [ ]:
# We can make a decision to drop the outlier, if we choose!

df = df[df['apps'] < 30000]
df.shape

# EDA: Create Histograms of All Numerical Columns
---

In [ ]:
df.hist(figsize=(15, 15));

# EDA: Plot a Heatmap of the Correlation Matrix
---

Heatmaps are an effective way to visually examine the correlational structure of your predictors. 

In [ ]:
plt.figure(figsize=(15,15))
sns.heatmap(df.corr(numeric_only=True), 
            annot=True,
           vmin = -1,
           vmax = 1,
           cmap = 'coolwarm');

##### What if I just wanted to look at the correlation to `apps`?

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(numeric_only=True)[['apps']].sort_values(by='apps', ascending=False), 
            annot=True,
           vmin=-1,
           vmax=1,
           cmap = 'coolwarm');

# EDA: Use seaborn's `.pairplot()` method to create scatterplots 
---

Let's create a pairplot to see how some of our stronger predictors correlate to our target (`apps`). Instead of creating a pairplot of the entire DataFrame, we can use the `y_vars` and `x_vars` params to get a smaller subset.

In [ ]:
sns.pairplot(df, x_vars=['accept', 'enroll', 'phd', 'f_undergrad', 'perc_alumni'], y_vars=['apps']);

In [ ]:
df.hist(figsize=(15, 15));

# Model Prep: Create our features matrix (`X`) and target vector (`y`)
---

What should our features be?

The `apps` column is our label: the number of applications received by that university.

In the cell below, create your `X` and `y` variables.

In [ ]:
X = df.drop(columns=['university', 'accept', 'enroll', 'apps'])
y = df['apps']

# Model Prep: Train/test split
---

We always want to have a holdout set to test our model. Use the `train_test_split` function to split our `X` and `y` variables into a training set and a holdout set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# Model Prep: Scaling
---

In the cell below, fit a `StandardScaler` to `X_train` and use it to transform both `X_train` and `X_test`. This is especially important when we learn about `Ridge` and `Lasso` next week!

In [ ]:
ss = StandardScaler()

X_train = ss.fit_transform(X_train)
X_test = ss.transform(X_test)

# Model Prep: Instantiate our model
---


In [ ]:
lr = LinearRegression()

# Evaluation
---

Use `.score` to evaluate the model.

In [ ]:
lr.fit(X_train, y_train)

print(f"Training R2 = {lr.score(X_train, y_train)}")
print(f"Testing R2 = {lr.score(X_test, y_test)}")

# LINE Assumptions
---


In [ ]:
# N - Normality of errors
y_pred = lr.predict(X_train)
resids = y_train - y_pred

plt.figure(figsize = (8, 6))
plt.hist(resids, bins = 25, color = 'teal', edgecolor = 'black');

# Fine

In [ ]:
# E - Equal variance of errors
# Residual plot
plt.figure(figsize = (10, 7))
plt.scatter(y_pred, resids, color = 'teal', edgecolor = 'black')
plt.axhline(0, color = 'black');

# Heteroscedasticity (!!)

> Variance of the dependent variable varies and is not constant across the entire dataset.

In [ ]:
 # What do we do? One option is to transform our dependent variable. A common transformation is to simply take the log of the dependent variable.

In [ ]:
X = df.drop(columns=['university', 'accept', 'enroll', 'apps'])
y_log = np.log(df['apps'])

In [ ]:
# What did this just do?
plt.hist(y, bins = 25, color = 'darkred', edgecolor = 'black')
plt.title('Distribution of `apps`');

In [ ]:
plt.hist(y_log, bins = 25, color = 'darkred', edgecolor = 'black')
plt.title('Distribution of log(`apps`)');

## **Note**
> Logarithmic transformation is a convenient means of transforming a highly skewed variable into a more normalized dataset.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y_log, random_state = 42)

In [ ]:
ss = StandardScaler()

X_train = ss.fit_transform(X_train)
X_test = ss.transform(X_test)

In [ ]:
lr_log = LinearRegression()
lr_log.fit(X_train, y_train)

In [ ]:
y_pred = lr_log.predict(X_train)
resids = y_train - y_pred

# E - Equal variance of errors
# Residual plot
plt.figure(figsize = (10, 7))
plt.scatter(y_pred, resids, color = 'teal', edgecolor = 'black')
plt.axhline(0, color = 'black');

In [ ]:
lr_log.score(X_train, y_train)

In [ ]:
lr_log.score(X_test, y_test)

### 😔

## Let's say that this model DID actually perform better... what do I need to remember to do to make predictions?

In [ ]:
# You must use the exponential to "undo" our log transformation!
log_preds = lr_log.predict(X_train)
preds = np.exp(log_preds)

In [ ]:
preds